<div align="center">
    <img src="https://www.sharif.ir/documents/20124/0/logo-fa-IR.png/4d9b72bc-494b-ed5a-d3bb-e7dfd319aec8?t=1609608338755" alt="Logo" width="200">
    <p><b>Sharif University of Technology</b></p>
    <p>Deep Learning Course, Dr. Soleymani</p>
    <p>Spring 2026</p>
</div>

---


*Full Name:*

*Student ID:*

# Retrieval-Augmented Generation (RAG) – From Scratch to Conversation

## Overview

**Retrieval-Augmented Generation (RAG)** is a technique that enhances a language model’s output by first retrieving relevant information from a knowledge base, then conditioning the generation on that retrieved evidence.

Mathematically, given a query $x$, we want to model:

$
p(y \mid x) = \sum_{z \in \text{Top-}k(x)} p_{\text{retrieve}}(z \mid x) \; p_{\text{generate}}(y \mid x, z)
$

where $z$ is a retrieved document chunk, often treated as a hard selection after top‑k retrieval.

### Dataset used in this notebook

Instead of a manually written toy knowledge base, this version uses a small real subset of **SQuAD v1** from Hugging Face. SQuAD provides real Wikipedia-based contexts, questions, and gold answers. To keep the notebook lightweight, we only use a small validation subset.

### In this notebook you will:

1. **From scratch (no LangChain)**:
   - Load a small real QA dataset
   - Chunk real Wikipedia contexts
   - Create dense embeddings with SentenceTransformers
   - Build a FAISS index for fast nearest-neighbour search
   - Implement a retriever and a GPT‑2 generator
   - Assemble a complete, stateless RAG system

2. **With LangChain**:
   - Integrate memory for multi-turn conversations
   - Implement a **history-aware** retriever that rewrites follow-up questions

3. **With an encoder-decoder generator**:
   - Replace GPT‑2 with FLAN‑T5
   - Compare decoder-only RAG and encoder-decoder RAG on the same retrieved contexts

After completing this notebook, you will understand the core components of RAG, the need for memory in conversational systems, and the architectural difference between decoder-only and encoder-decoder generators.

**Dependencies**: Run the cell below to install required packages.

In [45]:
# Install necessary packages (if not already installed)
import sys
!{sys.executable} -m pip install --quiet torch transformers sentence-transformers faiss-cpu datasets langchain-community langchain-huggingface langchain-core
# We use:
# - datasets for loading a small real SQuAD subset
# - sentence-transformers for dense embeddings
# - faiss-cpu for vector search
# - transformers for GPT-2 and FLAN-T5 generators
# - langchain-community / langchain-huggingface for the vectorstore wrapper in the conversational part


In [46]:
import os
import json
import torch
import numpy as np
import pandas as pd
from typing import List, Tuple, Dict

# For reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## Part 1: Document Loading & Text Chunking

A RAG system needs a collection of documents to retrieve from. In real applications these are often large texts, so we must split them into manageable **chunks**. Each chunk is a unit that can be embedded and later retrieved.

In this notebook, we use a small real subset of **SQuAD v1** instead of a manually written toy text file. Each SQuAD example contains:

- a real Wikipedia paragraph as `context`,
- a `question`,
- one or more gold `answers`.

We use the contexts as the retrieval corpus and the questions/gold answers for testing.

**Why chunk?**
- Language models have a limited context window.
- Smaller chunks allow more precise retrieval; a long document may contain many topics.

**Chunking trade‑offs**:
- Too small → loss of surrounding context.
- Too large → retrieval may bring irrelevant detail, and generation may exceed token limits.

**Implementation**: We’ll implement a simple recursive character‑based splitter with user‑defined chunk size and overlap.

### Questions 1
1. What would happen if we used whole documents without chunking?  
2. How does the overlap parameter help preserve continuity between chunks?

In [47]:
# TODO: Implement `chunk_text`.
# Split text into overlapping character chunks and return a list of strings.
# Expected output is kept below as a reference.


Number of chunks: 34
First chunk length: 100


In [48]:
# TODO: Load a small real SQuAD subset using `load_dataset`.
# Then extract unique contexts and create `chunks` using `chunk_text`.
# Expected output is kept below as a reference.


Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 300
})
Example row:
{'id': '56be4db0acb8001400a502ec', 'title': 'Super_Bowl_50', 'context': 'Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi\'s Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniversary" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as "Super Bowl L"), so that the logo could prominently feature the Arabic numerals 50.', 'question': 'Which NFL team represented

## Part 2: Document Embeddings & Vector Index

To retrieve relevant chunks, we need to embed both the documents and the query into a common dense vector space. We use a pre‑trained sentence transformer to obtain **fixed‑size embeddings**.

**Dense retrieval** computes:

$
\text{sim}(q, d) = \cos(\phi(q), \phi(d)) \quad \text{or} \quad \|\phi(q) - \phi(d)\|_2
$

where $\phi$ is the embedding function.

We’ll store the embeddings in a **FAISS** index for fast approximate (or exact) nearest‑neighbour search.

### Questions 2
1. Explain the difference between sparse retrieval (e.g., TF‑IDF, BM25) and dense retrieval.  
2. Why is L2 distance often used instead of cosine similarity in FAISS `IndexFlatL2`? (Hint: embedding normalisation)

In [49]:
# TODO: Load `all-MiniLM-L6-v2` with SentenceTransformer.
# Encode all chunks into float32 dense embeddings.
# Expected output is kept below as a reference.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding dimension: 384


/tmp/ipykernel_58/2764334989.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Embedding dimension: {embed_model.get_sentence_embedding_dimension()}")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings shape: (30, 384)


In [50]:
# TODO: Build a FAISS IndexFlatL2 index.
# Add all chunk embeddings to the index.
# Expected output is kept below as a reference.


FAISS index contains 30 vectors.


## Part 3: The Retriever

The retriever is responsible for, given a user query, finding the top‑k most relevant chunks.

Formally:

$
\text{Retrieve}(q, k) = \text{argtopk}_{d \in \mathcal{D}} \; \text{sim}(\phi(q), \phi(d))
$

where $\mathcal{D}$ is the set of all chunk embeddings.

### Question 3
How would you modify the retriever to use **maximum inner product search** (MIPS) instead of L2 distance? When would MIPS be preferred?

In [51]:
# TODO: Implement `retrieve(query, index, embed_model, chunks, top_k)`.
# Embed the query, search FAISS, and return top-k (chunk, distance) pairs.
# Expected output is kept below as a reference.


Question: Which NFL team represented the AFC at Super Bowl 50?
Gold answer: Denver Broncos

Top retrieved chunks:

Distance 0.6532:
Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi's Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniv ...

Distance 0.6940:
hey joined the Patriots, Dallas Cowboys, and Pittsburgh Steelers as one of four teams that have made eight appearances in the Super Bowl. ...

Distance 0.7646:
The Panthers finished the regular season with a 15–1 record, and quarterback Cam Newton was named the NFL Most Valuable Player (MVP). They defeated the Arizona Cardinals 49–15 in the NF

## Part 4: The Generator (GPT‑2)

We use a pre‑trained auto‑regressive language model (GPT‑2) to generate an answer conditioned on the retrieved context and the question.

**Input format** we will use:

Context:

chunk 1

chunk 2
...

Question: {user query}

Answer:

GPT‑2 will then continue the string.

### Questions 4
1. Why do we concatenate the context and the question?  
2. What are the limitations of using a base GPT‑2 model for knowledge‑intensive generation?

In [52]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

gpt2_model_name = "gpt2"  # You can also try "gpt2-medium" if resources allow.
tokenizer = GPT2Tokenizer.from_pretrained(gpt2_model_name)
model = GPT2LMHeadModel.from_pretrained(gpt2_model_name)
model.to(device)
model.eval()

# GPT-2 has no pad token by default; set it to eos for generation.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [53]:
# TODO: Implement `build_rag_prompt` and `generate_answer` for GPT-2.
# Concatenate retrieved context and question, then generate an answer.
# Expected output is kept below as a reference.


Question: Which NFL team represented the AFC at Super Bowl 50?
Gold answer: Denver Broncos
GPT-2 answer: The Denver Broncos.
.
 (Note: The Super Bowl was played in the United States, not the United Kingdom.)


## Part 5: Putting It All Together – Stateless RAG

Now we combine the retriever and the generator into a single pipeline.

Given a question:
1. Retrieve top‑k chunks.
2. Concatenate them into a context string.
3. Feed context + question to the GPT‑2 generator.

Let’s implement a `RAGSystem` class that encapsulates this process.

### Question 5
This system answers each question independently. What problem arises when a user asks a follow‑up question like *“Tell me more about that”* or *“Multiply the previous answer by 2”*?

In [54]:
# TODO: Implement the stateless `RAGSystem` class.
# It should retrieve chunks, build the context, and generate an answer.
# Expected output is kept below as a reference.


Question: Which NFL team represented the AFC at Super Bowl 50?
Gold answer: Denver Broncos

Retrieved chunks:
- Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super B ...
- hey joined the Patriots, Dallas Cowboys, and Pittsburgh Steelers as one of four teams that have made eight appearances in the Super Bowl. ...
- The Panthers finished the regular season with a 15–1 record, and quarterback Cam Newton was named the NFL Most Valuable Player (MVP). They defeated the Arizona Cardinals 49–15 in the NFC Championship Game and advanced to their second Super Bowl appearance since the franchise was founded in 1995. The ...

GPT-2 RAG answer: The New England Revolution.
.
 (Note: The Patriots were the only team to win the Superbowl in Super Bowl 51.)


## Part 6: The Memory Problem

Let’s simulate a conversation where the second question depends on the first one.

**Stateless RAG will fail or become unstable** because each call is independent. The retriever only sees the latest question, so a follow-up such as *“Where was it played?”* or *“Which team represented the NFC in that game?”* is ambiguous unless the system remembers that the previous topic was **Super Bowl 50**.


In [55]:
q1 = qa_dataset[0]["question"]
ans1 = rag.query(q1, top_k=3, max_new_tokens=80)
print(f"Q: {q1}\nA: {ans1['answer']}\n")

# Follow-up that refers to the previous topic.
# Without memory, the phrase "that game" is ambiguous for the retriever.
q2 = "Which team represented the NFC in that game?"
ans2 = rag.query(q2, top_k=3, max_new_tokens=80)
print(f"Q: {q2}\nA: {ans2['answer']}\n")

# Notice: the second answer may be weaker because stateless RAG does not explicitly
# remember that the previous question was about Super Bowl 50.

Q: Which NFL team represented the AFC at Super Bowl 50?
A: The New England Revolution.
.
 (Note: The Patriots were the only team to win the Superbowl in Super Bowl 51.)

Q: Which team represented the NFC in that game?
A: The New York Giants.
.
 (Note: The Giants were the only team to win the Superbowl in the NFC Championship Game.)



## Part 7: Adding Memory with LangChain

To enable multi-turn conversations, we need to **maintain a chat history** and use it to reformulate the current question into a standalone query.

In many LangChain tutorials, this is done with functions such as `create_history_aware_retriever` and `create_retrieval_chain`. However, these imports can break across different LangChain versions. To keep this notebook stable, we implement the same logic explicitly while still using LangChain for the FAISS vectorstore wrapper.

We will:

1. Wrap the same FAISS index and embedding model with LangChain’s `FAISS` vectorstore.
2. Store the conversation in a simple `chat_history` list.
3. Rewrite ambiguous follow-up questions into standalone questions.
4. Retrieve documents using the standalone question.
5. Generate the final answer using **FLAN-T5**, which is much better for instruction following than base GPT-2.

### Questions 6
1. Explain how a history-aware retriever works internally. What prompt or rewriting step does it use?  
2. Why can’t we simply pass the whole chat history to the retriever instead of generating a standalone question?


In [56]:
# TODO: Create LangChain `Document` objects from chunks.
# Then build a FAISS vectorstore using HuggingFace embeddings.
# Expected output is kept below as a reference.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vectorstore contains 30 documents.


In [57]:
# TODO: Load `google/flan-t5-small` as an encoder-decoder model.
# Implement a small helper function to generate text with FLAN-T5.
# Expected output is kept below as a reference.


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [58]:
# TODO: Implement helper functions for conversational RAG.
# Convert chat history to text and rewrite short follow-up questions.


In [59]:
# TODO: Implement `SimpleConversationalRAG`.
# Steps: rewrite follow-up question, retrieve documents, generate answer.


In [60]:
# Test the corrected memory-based RAG pipeline.
chat_history = []

question_1 = "What was Super Bowl 50?"
response_1 = rag_chain_with_memory.invoke({
    "input": question_1,
    "chat_history": chat_history
})

print("Question 1:", question_1)
print("Standalone question 1:", response_1["standalone_question"])
print("Answer 1:", response_1["answer"])

chat_history.append(("human", question_1))
chat_history.append(("ai", response_1["answer"]))

question_2 = "Where was it played?"
response_2 = rag_chain_with_memory.invoke({
    "input": question_2,
    "chat_history": chat_history
})

print("\nQuestion 2:", question_2)
print("Standalone question 2:", response_2["standalone_question"])
print("Answer 2:", response_2["answer"])
print("\nRetrieved context for turn 2:")
print(response_2["retrieved_context"][:1000], "...")


Question 1: What was Super Bowl 50?
Standalone question 1: What was Super Bowl 50?
Answer 1: an American football game to determine the champion

Question 2: Where was it played?
Standalone question 2: Where was Super Bowl 50 played?
Answer 2: Santa Clara, California

Retrieved context for turn 2:
Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi's Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniversary" with various gold-themed initiatives, as well as temporarily suspending the tradition of nam

On May 21, 2013, NFL owners at their spring meetings in Boston voted and awarded the game to Levi's

In [61]:
from langchain_core.messages import HumanMessage, AIMessage

chat_history = []


def ask_question(query: str):
    response = rag_chain_with_memory.invoke({
        "input": query,
        "chat_history": chat_history
    })

    # Update chat history after each turn.
    chat_history.append(HumanMessage(content=query))
    chat_history.append(AIMessage(content=response["answer"]))
    return response


# First turn: a real SQuAD-style question.
q1 = qa_dataset[0]["question"]
response1 = ask_question(q1)
print("User:", q1)
print("Standalone:", response1["standalone_question"])
print("Assistant:", response1["answer"])

# Follow-up question that depends on the previous topic.
q2 = "Which team represented the NFC in that game?"
response2 = ask_question(q2)
print("\nUser:", q2)
print("Standalone:", response2["standalone_question"])
print("Assistant:", response2["answer"])


User: Which NFL team represented the AFC at Super Bowl 50?
Standalone: Which NFL team represented the AFC at Super Bowl 50?
Assistant: Denver Broncos

User: Which team represented the NFC in that game?
Standalone: Which team represented the NFC in Super Bowl 50?
Assistant: Carolina Panthers.


### Explanation for Question 6

A history-aware retriever first checks whether the latest user question is self-contained. If the question contains references such as *it*, *that game*, or *the previous answer*, the system uses the chat history to rewrite it into a standalone query. For example:

```text
Chat history: What was Super Bowl 50?
Follow-up: Where was it played?
Standalone query: Where was Super Bowl 50 played?
```

The retriever should usually receive this standalone query rather than the whole chat history. Passing the entire chat history directly to the retriever can introduce irrelevant words from previous turns and make vector search less focused. A short standalone query preserves the missing context while keeping retrieval precise.

In this notebook, we implement the history-aware logic explicitly instead of relying on `langchain.chains`, because those imports are version-sensitive. The idea is the same: use memory to rewrite the query, retrieve relevant documents, then answer from the retrieved context.


## Part 8: RAG with an Encoder-Decoder Generator

So far, the retriever has been encoder-based, but the generator has been GPT‑2, which is a decoder-only language model.

In this part, we replace GPT‑2 with an encoder-decoder model, **google/flan-t5-small**. The retrieved context and the user question are passed to the encoder, and the decoder generates the answer.

### Question 7
Implement a RAG pipeline that uses the same retriever as before, but replaces GPT‑2 with an encoder-decoder model such as `google/flan-t5-small`.

1. Compare the GPT‑2-based RAG and the T5-based RAG on at least two questions.
2. Explain how the retrieved context is used differently in GPT‑2 and T5.
3. Why is T5 considered an encoder-decoder model, while GPT‑2 is decoder-only?

In [62]:
# TODO: Implement the T5-based RAG pipeline.
# Use the same retriever, but replace GPT-2 with FLAN-T5.
# Expected output is kept below as a reference.


Question: Which NFL team represented the AFC at Super Bowl 50?
Gold answer: Denver Broncos

Retrieved context:
 Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi's Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniversary" with various gold-themed initiatives, as well as temporarily suspending the tradition of nam

hey joined the Patriots, Dallas Cowboys, and Pittsburgh Steelers as one of four teams that have made eight appearances in the Super Bowl.

The Panthers finished the regular season with a 15–1 record, and quarterback Cam Newton was named the NFL Most Valuable Player (MVP). They defeated

In [63]:
# TODO: Compare GPT-2 RAG and T5 RAG on at least two questions.
# Use the same retrieved context for both models for a fair comparison.
# Expected output is kept below as a reference.


,Question,Gold Answer,GPT-2 RAG Answer,T5 Encoder-Decoder RAG Answer
0,Which NFL team represented the AFC at Super Bo...,Denver Broncos,The New England Revolution.\n.\n (Note: The Pa...,Denver Broncos
1,What was the theme of Super Bowl 50?,"""golden anniversary""",The Golden Age of the league was the first tim...,"""golden anniversary"


Question: Which NFL team represented the AFC at Super Bowl 50?
Gold answer: Denver Broncos

GPT-2 RAG Answer:
The New England Revolution.
.
 (Note: The Patriots were the only team to win the Superbowl in Super Bowl 51.)

T5 Encoder-Decoder RAG Answer:
Denver Broncos

Retrieved Context:
Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi's Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniversary" with various gold-themed initiatives, as well as temporarily suspending the tradition of nam

hey joined the Patriots, Dallas Cowboys, and Pittsburgh Steelers as one of four teams that have made eight appe

### Explanation for Question 7

In both systems, the retriever is exactly the same. The question is embedded using the SentenceTransformer model, and FAISS retrieves the top-k most relevant chunks from the real SQuAD context corpus.

The main difference is the generator architecture.

In the **GPT‑2-based RAG system**, the retrieved context, the question, and the `Answer:` prefix are concatenated into one prompt. GPT‑2 is a **decoder-only** model, so it generates the answer by continuing this prompt from left to right. During generation, each new token can attend only to previous tokens, including the retrieved context and the question.

In the **T5-based RAG system**, the retrieved context and the question are passed to the **encoder**. The encoder reads the full input sequence and produces contextual representations. Then the **decoder** generates the answer autoregressively while attending to the encoder outputs through cross-attention. Therefore, the retrieved context is used as an encoded source sequence rather than only as a text prefix.

T5 is considered an **encoder-decoder** model because it has two separate Transformer components: an encoder for processing the input text and a decoder for generating the output text. GPT‑2 is **decoder-only** because it only contains the autoregressive Transformer decoder stack and generates text by predicting the next token from the previous tokens.

In practice, FLAN‑T5 usually gives cleaner answers for this task than base GPT‑2 because FLAN‑T5 is instruction-tuned and better aligned with question answering. Base GPT‑2 is mainly trained for next-token prediction, so it may continue the prompt instead of directly answering the question.

## Summary

You have:
- Built a **stateless RAG system** from real SQuAD contexts, embeddings, FAISS, and GPT‑2.
- Observed its weakness on follow-up questions.
- Used a LangChain FAISS vectorstore and an explicit **history-aware query rewriting** step to add conversational memory.
- Replaced the unstable GPT‑2 memory generator with **FLAN‑T5** for reformulation and answering.
- Replaced the decoder-only GPT‑2 generator with an **encoder-decoder FLAN‑T5** generator.
- Compared GPT‑2-based RAG and T5-based RAG on the same retrieved contexts.

### Final Theoretical Questions
1. Compare the stateless and conversational RAG pipelines. What are the main differences in the retrieval step?
2. In the conversational version, where is the ‘memory’ actually stored?
3. How could you extend this system to handle **multiple users**? What would you need to change?
4. Suppose you wanted to replace GPT‑2 with a model that does not support the “reformulate a standalone question” task well. What alternatives could you explore?
5. Why can an encoder-decoder model such as T5 be more suitable for question answering than a base decoder-only model such as GPT‑2?

### References
- [SQuAD: 100,000+ Questions for Machine Comprehension of Text](https://rajpurkar.github.io/SQuAD-explorer/)
- [Lewis et al., Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks (2020)](https://arxiv.org/abs/2005.11401)
- [FAISS library](https://github.com/facebookresearch/faiss)
- [FLAN-T5 model family](https://huggingface.co/google/flan-t5-small)
